# Data Preprocessing

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import joblib

In [2]:
PROJECT_ROOT = Path("..")

DATASET_PATH = PROJECT_ROOT / "datasets" / "processed" / "processed_train.csv"

MODEL_DIR = PROJECT_ROOT / "models"

MODEL_DIR.mkdir(exist_ok=True)

In [3]:
df = pd.read_csv(DATASET_PATH)

df.head()

,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,2015/9/4,Tas,Ryegrass_Clover,0.62,4.6667,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,2015/4/1,NSW,Lucerne,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,2015/9/1,WA,SubcloverDalkeith,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,2015/5/18,Tas,Ryegrass,0.66,5.0000,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,2015/9/11,Tas,Ryegrass,0.54,3.5000,0.4343,23.2239,10.5261,34.1844,10.9605


In [4]:
print(df.shape)

df.info()

(357, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 357 entries, 0 to 356
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   image_path     357 non-null    object 
 1   Sampling_Date  357 non-null    object 
 2   State          357 non-null    object 
 3   Species        357 non-null    object 
 4   Pre_GSHH_NDVI  357 non-null    float64
 5   Height_Ave_cm  357 non-null    float64
 6   Dry_Clover_g   357 non-null    float64
 7   Dry_Dead_g     357 non-null    float64
 8   Dry_Green_g    357 non-null    float64
 9   Dry_Total_g    357 non-null    float64
 10  GDM_g          357 non-null    float64
dtypes: float64(7), object(4)
memory usage: 30.8+ KB


Define Fetaure grp

In [5]:
image_column = "image_path"

categorical_columns = [
    "State",
    "Species"
]

numerical_columns = [
    "NDVI",
    "Height_cm"
]

target_columns = [
    "Dry_Green_g",
    "Dry_Dead_g",
    "Dry_Clover_g",
    "Dry_Total_g",
    "GDM_g"
]

Lets encode categories

In [6]:
label_encoders = {}

for column in categorical_columns:

    encoder = LabelEncoder()

    df[column] = encoder.fit_transform(df[column])

    label_encoders[column] = encoder

In [7]:
df.head()

,image_path,Sampling_Date,State,Species,Pre_GSHH_NDVI,Height_Ave_cm,Dry_Clover_g,Dry_Dead_g,Dry_Green_g,Dry_Total_g,GDM_g
0,train/ID1011485656.jpg,2015/9/4,1,11,0.62,4.6667,0.0000,31.9984,16.2751,48.2735,16.2750
1,train/ID1012260530.jpg,2015/4/1,0,3,0.55,16.0000,0.0000,0.0000,7.6000,7.6000,7.6000
2,train/ID1025234388.jpg,2015/9/1,3,12,0.38,1.0000,6.0500,0.0000,0.0000,6.0500,6.0500
3,train/ID1028611175.jpg,2015/5/18,1,10,0.66,5.0000,0.0000,30.9703,24.2376,55.2079,24.2376
4,train/ID1035947949.jpg,2015/9/11,1,10,0.54,3.5000,0.4343,23.2239,10.5261,34.1844,10.9605


Scale the numericals features

In [8]:
print(df.columns.tolist())

['image_path', 'Sampling_Date', 'State', 'Species', 'Pre_GSHH_NDVI', 'Height_Ave_cm', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']


In [9]:
print(numerical_columns)

['NDVI', 'Height_cm']


In [10]:
# Rename columns for consistency

df = df.rename(columns={
    "Pre_GSHH_NDVI": "NDVI",
    "Height_Ave_cm": "Height_cm"
})

In [11]:
scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(
    df[numerical_columns]
)

In [12]:
print(df.head())

print("\nCategorical columns:")
print(df[["State", "Species"]].head())

print("\nNumerical columns:")
print(df[["NDVI", "Height_cm"]].describe())

               image_path Sampling_Date  State  Species      NDVI  Height_cm  \
0  train/ID1011485656.jpg      2015/9/4      1       11 -0.246319  -0.285204   
1  train/ID1012260530.jpg      2015/4/1      0        3 -0.707060   0.818240   
2  train/ID1025234388.jpg      2015/9/1      3       12 -1.826004  -0.642205   
3  train/ID1028611175.jpg     2015/5/18      1       10  0.016962  -0.252753   
4  train/ID1035947949.jpg     2015/9/11      1       10 -0.772880  -0.398797   

   Dry_Clover_g  Dry_Dead_g  Dry_Green_g  Dry_Total_g    GDM_g  
0        0.0000     31.9984      16.2751      48.2735  16.2750  
1        0.0000      0.0000       7.6000       7.6000   7.6000  
2        6.0500      0.0000       0.0000       6.0500   6.0500  
3        0.0000     30.9703      24.2376      55.2079  24.2376  
4        0.4343     23.2239      10.5261      34.1844  10.9605  

Categorical columns:
   State  Species
0      1       11
1      0        3
2      3       12
3      1       10
4      1       10

Save the preprocessing artifacts

In [13]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
joblib.dump(label_encoders, MODEL_DIR / "label_encoders.pkl")

print("Preprocessing artifacts saved successfully.")

Preprocessing artifacts saved successfully.


Now,
Train/Validation split

In [14]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print("Training samples :", len(train_df))
print("Validation samples :", len(val_df))

Training samples : 285
Validation samples : 72


Save the processed dataset

In [15]:
processed_dir = PROJECT_ROOT / "datasets" / "processed"

train_df.to_csv(
    processed_dir / "train_processed.csv",
    index=False
)

val_df.to_csv(
    processed_dir / "val_processed.csv",
    index=False
)

print("Datasets saved successfully.")

Datasets saved successfully.


In [16]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [17]:
from src.data.transforms import train_transform

print(train_transform)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=warn)
    RandomHorizontalFlip(p=0.5)
    RandomRotation(degrees=[-10.0, 10.0], interpolation=nearest, expand=False, fill=0)
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.8, 1.2), hue=(-0.05, 0.05))
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [18]:
df = pd.read_csv("../datasets/processed/train_processed.csv")

print(df.columns.tolist())

['image_path', 'Sampling_Date', 'State', 'Species', 'NDVI', 'Height_cm', 'Dry_Clover_g', 'Dry_Dead_g', 'Dry_Green_g', 'Dry_Total_g', 'GDM_g']


In [19]:
# Convert Sampling_Date to datetime
df["Sampling_Date"] = pd.to_datetime(df["Sampling_Date"])

In [20]:
# Extract month (1-12)
df["Month"] = df["Sampling_Date"].dt.month

In [21]:
df = df.drop(columns=["Sampling_Date"])

In [22]:
numerical_features = [
    "NDVI",
    "Height_cm",
    "Month"
]

In [23]:
column_order = [
    "image_path",
    "State",
    "Species",
    "NDVI",
    "Height_cm",
    "Month",
    "Dry_Clover_g",
    "Dry_Dead_g",
    "Dry_Green_g",
    "Dry_Total_g",
    "GDM_g"
]

df = df[column_order]

In [24]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

In [25]:
column_order = [
    "image_path",
    "State",
    "Species",
    "NDVI",
    "Height_cm",
    "Month",
    "Dry_Clover_g",
    "Dry_Dead_g",
    "Dry_Green_g",
    "Dry_Total_g",
    "GDM_g"
]

train_df = train_df[column_order]
val_df = val_df[column_order]

In [26]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

processed_dir = PROJECT_ROOT / "datasets" / "processed"

train_df.to_csv(
    processed_dir / "train_processed.csv",
    index=False
)

val_df.to_csv(
    processed_dir / "val_processed.csv",
    index=False
)

print("✅ Processed datasets saved successfully!")

✅ Processed datasets saved successfully!


# Testing Parts here

In [27]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data.dataset import BiomassDataset
from src.data.transforms import train_transform

dataset = BiomassDataset(
    csv_file="../datasets/processed/train_processed.csv",
    transform=train_transform
)

print("Dataset size:", len(dataset))

Dataset size: 228


In [28]:
sample = dataset[0]

print(sample.keys())

print(sample["image"].shape)

print(sample["metadata"])

print(sample["targets"])

dict_keys(['image', 'metadata', 'targets'])
torch.Size([3, 224, 224])
tensor([ 0.9384, -0.3014, 11.0000,  1.0000,  0.0000])
tensor([ 3.7880, 12.3111, 35.0394, 51.1386, 38.8275])


In [29]:
from src.config import NUMERICAL_COLUMNS

print(NUMERICAL_COLUMNS)

['NDVI', 'Height_cm', 'Month']


In [30]:
from src.config import CATEGORICAL_COLUMNS

print(CATEGORICAL_COLUMNS)

['State', 'Species']


# here bit of problem happen due to old cache storage in the jupyter notebook 
Hence the code might look messy because i somewhere used the code twice to just resolve the problem